# Fixed-rate ramp-metering policy on the exact corrected CTM

One constant meter command per ramp, held for all 480 steps, evaluated through the **exact corrected benchmark plant** (`Benchmark_calculation` v3) under the official 1.2-arrival scenario.

Rules enforced here:
1. A fixed policy means `u_r,k = c_r` for all k — one constant rate per ramp (4 decisions), never a scaled copy of the PeMS flow profile.
2. Everything (CTM equations, initial state, arrivals, boundary/external demand, off-ramp behavior, objective) is loaded from the corrected benchmark; nothing is redefined locally.
3. The wrapper is regression-tested: commanding the benchmark's fully-open constants must reproduce the official benchmark exactly.
4. All physical invariants (sending, receiving, merge, four mass ledgers) and the objective-decomposition identity are hard-asserted for every reported policy.
5. The optimized result is the *best fixed-rate vector found* (derivative-free search over a nonconvex exact objective), not a certified global optimum.


In [1]:
# 1. Load the exact corrected benchmark.

from pathlib import Path
import time

import numpy as np
import pandas as pd
from scipy.optimize import (
    differential_evolution,
    minimize,
)
from IPython.display import display
from IPython.utils.capture import capture_output

pd.set_option(
    "display.max_columns",
    120
)

pd.set_option(
    "display.max_rows",
    120
)

BENCHMARK_NOTEBOOK = Path(
    "Benchmark_calculation.ipynb"
)

if not BENCHMARK_NOTEBOOK.exists():
    raise FileNotFoundError(
        f"{BENCHMARK_NOTEBOOK} was not found."
    )

# Execute the exact corrected benchmark.
# Output is suppressed, but exceptions are not suppressed.
with capture_output():
    get_ipython().run_line_magic(
        "run",
        f'"{BENCHMARK_NOTEBOOK}"'
    )

if "shared_benchmark_inputs" not in globals():
    raise RuntimeError(
        "The corrected benchmark did not define "
        "shared_benchmark_inputs."
    )

required_functions = [
    "simulate_state_based_benchmark_480_steps",
    "compute_mainline_mass_residual",
    "compute_upstream_boundary_mass_residual",
    "compute_external_entry_mass_residual",
    "compute_max_sending_violation",
    "compute_max_receiving_violation",
    "compute_max_merge_violation",
]

missing_functions = [
    function_name
    for function_name in required_functions
    if function_name not in globals()
]

if missing_functions:
    raise RuntimeError(
        "Missing corrected benchmark functions: "
        f"{missing_functions}"
    )

S = shared_benchmark_inputs

required_keys = [
    "num_steps",
    "delta_t",
    "arrival_multiplier",
    "mainline_initial_state",
    "ramp_queue_0",
    "ramp_overflow_queue_0",
    "external_entry_queue_0",
    "q_in_boundary_series",
    "ramp_arrival_series",
    "external_inflow_series",
    "fixed_outflow_series",
    "inflow_capacity",
    "outflow_capacity",
    "physical_capacity",
    "safe_threshold_capacity",
    "movement_factor_by_cell",
    "wave_speed_ratio_by_cell",
    "exit_split_by_cell",
    "ramp_ids",
    "ramp_cell_map",
    "ramp_name_map",
    "ramp_max_queue_by_u",
    "ramp_max_queue_named",
    "tt_ff_min",
    "AW",
    "gamma",
    "lambda_safe",
    "lambda_spillback",
    "fully_open_command_by_ramp",
    "official_benchmark_history",
    "official_totals",
    "official_service_metrics",
]

missing_keys = [
    key
    for key in required_keys
    if key not in S
]

if missing_keys:
    raise KeyError(
        "Missing corrected benchmark keys: "
        f"{missing_keys}"
    )

cell_ids = [
    f"Cell {index}"
    for index in range(1, 10)
]

ramp_ids = list(
    S["ramp_ids"]
)

num_steps = int(
    S["num_steps"]
)

delta_t = float(
    S["delta_t"]
)

arrival_multiplier = float(
    S["arrival_multiplier"]
)

if num_steps != 480:
    raise ValueError(
        f"Expected 480 steps, received {num_steps}."
    )

if abs(
    arrival_multiplier
    - 1.2
) > 1e-12:
    raise ValueError(
        "Official arrival multiplier must be 1.2."
    )

expected_ramp_cell_map = {
    "u_4th": "Cell 2",
    "u_price": "Cell 2",
    "u_mattie": "Cell 6",
    "u_avila": "Cell 9",
}

if S["ramp_cell_map"] != expected_ramp_cell_map:
    raise AssertionError(
        "Ramp/cell mapping changed unexpectedly."
    )

AW = float(
    S["AW"]
)

gamma = float(
    S["gamma"]
)

lambda_safe = float(
    S["lambda_safe"]
)

lambda_spillback = float(
    S["lambda_spillback"]
)

print(
    "Corrected benchmark loaded."
)

print(
    "Benchmark objective:",
    S["official_totals"]["raw_objective"]
)


Corrected benchmark loaded.
Benchmark objective: 88338.75071339465


In [2]:
# 2. True constant-rate policy definition.

def validate_rate_vector(
    rate_vector
):
    rate_vector = np.asarray(
        rate_vector,
        dtype=float
    )

    if rate_vector.shape != (
        len(ramp_ids),
    ):
        raise ValueError(
            "The fixed-rate vector must contain "
            f"{len(ramp_ids)} values."
        )

    if np.any(
        ~np.isfinite(rate_vector)
    ):
        raise ValueError(
            "Fixed rates must be finite."
        )

    if np.any(
        rate_vector < 0.0
    ):
        raise ValueError(
            "Fixed rates cannot be negative."
        )

    return rate_vector


def rate_vector_to_dictionary(
    rate_vector
):
    rate_vector = validate_rate_vector(
        rate_vector
    )

    return {
        ramp: float(
            rate_vector[index]
        )
        for index, ramp in enumerate(
            ramp_ids
        )
    }


def build_constant_command_series(
    rate_vector
):
    rate_by_ramp = (
        rate_vector_to_dictionary(
            rate_vector
        )
    )

    command_series = {
        ramp: [
            rate_by_ramp[ramp]
            for _ in range(num_steps)
        ]
        for ramp in ramp_ids
    }

    # Hard check: each ramp command must be
    # constant through all 480 steps.
    for ramp in ramp_ids:
        time_variation = float(
            np.max(
                command_series[ramp]
            )
            - np.min(
                command_series[ramp]
            )
        )

        if time_variation > 1e-12:
            raise AssertionError(
                f"{ramp} command is not constant."
            )

    return command_series


In [3]:
# 3. Simulate the fixed policy through the exact CTM.

def simulate_constant_rate_policy(
    rate_vector
):
    command_series = (
        build_constant_command_series(
            rate_vector
        )
    )

    history = (
        simulate_state_based_benchmark_480_steps(
            num_steps=num_steps,

            mainline_initial_state=(
                S["mainline_initial_state"]
            ),

            ramp_queue_0=(
                S["ramp_queue_0"]
            ),

            q_in_boundary_series=(
                S["q_in_boundary_series"]
            ),

            commanded_release_series=(
                command_series
            ),

            # The official 1.2-arrival scenario is fixed.
            ramp_arrival_series=(
                S["ramp_arrival_series"]
            ),

            external_inflow_series=(
                S["external_inflow_series"]
            ),

            fixed_outflow_series=(
                S["fixed_outflow_series"]
            ),

            inflow_capacity=(
                S["inflow_capacity"]
            ),

            outflow_capacity=(
                S["outflow_capacity"]
            ),

            physical_capacity=(
                S["physical_capacity"]
            ),

            safe_threshold_capacity=(
                S["safe_threshold_capacity"]
            ),

            ramp_name_map=(
                S["ramp_name_map"]
            ),

            ramp_max_queue_named=(
                S["ramp_max_queue_named"]
            ),

            ramp_max_queue_by_u=(
                S["ramp_max_queue_by_u"]
            ),

            tt_ff_min=(
                S["tt_ff_min"]
            ),

            delta_t=delta_t,

            gamma=gamma,

            lambda_safe=lambda_safe,

            lambda_spillback=(
                lambda_spillback
            ),

            AW=AW,

            movement_factor_by_cell=(
                S["movement_factor_by_cell"]
            ),

            wave_speed_ratio_by_cell=(
                S["wave_speed_ratio_by_cell"]
            ),

            exit_split_by_cell=(
                S["exit_split_by_cell"]
            ),

            ramp_overflow_queue_0=(
                S["ramp_overflow_queue_0"]
            ),

            external_entry_queue_0=(
                S["external_entry_queue_0"]
            ),
        )
    )

    return history


In [4]:
# 4. Objective aggregation on the corrected components.

def aggregate_exact_history(
    history
):
    totals = {
        "mainline_tts": float(
            sum(
                history["mainline_tts"]
            )
        ),

        "ramp_queue_tts": float(
            sum(
                history["ramp_queue_tts"]
            )
        ),

        "upstream_boundary_tts": float(
            sum(
                history[
                    "upstream_boundary_tts"
                ]
            )
        ),

        "external_entry_tts": float(
            sum(
                history[
                    "external_entry_tts"
                ]
            )
        ),

        "fairness_penalty": float(
            sum(
                history[
                    "fairness_penalty"
                ]
            )
        ),

        "safe_exposure": float(
            sum(
                history[
                    "safe_exposure"
                ]
            )
        ),

        "spillback_exposure": float(
            sum(
                history[
                    "spillback_exposure"
                ]
            )
        ),

        "weighted_safe_penalty": float(
            sum(
                history[
                    "weighted_safe_penalty"
                ]
            )
        ),

        "weighted_spillback_penalty": float(
            sum(
                history[
                    "weighted_spillback_penalty"
                ]
            )
        ),

        "raw_objective": float(
            sum(
                history[
                    "total_objective"
                ]
            )
        ),
    }

    reconstructed_objective = (
        AW
        * totals["mainline_tts"]

        + totals["ramp_queue_tts"]

        + totals[
            "upstream_boundary_tts"
        ]

        + totals[
            "external_entry_tts"
        ]

        + totals[
            "fairness_penalty"
        ]

        + totals[
            "weighted_safe_penalty"
        ]

        + totals[
            "weighted_spillback_penalty"
        ]
    )

    totals[
        "objective_reconstruction_error"
    ] = (
        totals["raw_objective"]
        - reconstructed_objective
    )

    totals[
        "final_mainline_inventory"
    ] = sum(
        float(
            history["x_final"][cell]
        )
        for cell in cell_ids
    )

    totals[
        "final_ramp_queue"
    ] = sum(
        float(
            history["R_final"][ramp]
        )
        + float(
            history["B_final"][ramp]
        )
        for ramp in ramp_ids
    )

    totals[
        "final_upstream_boundary_queue"
    ] = float(
        history[
            "upstream_boundary_queue_final"
        ]
    )

    totals[
        "final_external_entry_queue"
    ] = sum(
        float(
            history[
                "external_entry_queue_final"
            ][cell]
        )
        for cell in cell_ids
    )

    totals[
        "final_total_inventory"
    ] = (
        totals[
            "final_mainline_inventory"
        ]

        + totals[
            "final_ramp_queue"
        ]

        + totals[
            "final_upstream_boundary_queue"
        ]

        + totals[
            "final_external_entry_queue"
        ]
    )

    initial_ramp_queue = sum(
        float(
            S["ramp_queue_0"][ramp]
        )
        + float(
            S["ramp_overflow_queue_0"][
                ramp
            ]
        )
        for ramp in ramp_ids
    )

    total_ramp_arrivals = sum(
        float(
            S["ramp_arrival_series"][
                ramp
            ][step]
        )
        for ramp in ramp_ids
        for step in range(num_steps)
    )

    total_actual_release = sum(
        float(
            history[
                "actual_release"
            ][step][ramp]
        )
        for ramp in ramp_ids
        for step in range(num_steps)
    )

    totals[
        "total_ramp_arrivals"
    ] = total_ramp_arrivals

    totals[
        "total_actual_release"
    ] = total_actual_release

    totals[
        "ramp_mass_residual"
    ] = (
        initial_ramp_queue
        + total_ramp_arrivals
        - total_actual_release
        - totals["final_ramp_queue"]
    )

    total_ramp_demand = (
        initial_ramp_queue
        + total_ramp_arrivals
    )

    if total_ramp_demand > 0.0:
        totals["served_fraction"] = (
            total_actual_release
            / total_ramp_demand
        )
    else:
        totals["served_fraction"] = 1.0

    return totals


In [5]:
# 5. Complete mathematical invariant checks.

def validate_fixed_policy_history(
    history,
    command_series,
    label
):
    totals = aggregate_exact_history(
        history
    )

    max_command_time_variation = max(
        float(
            np.max(
                command_series[ramp]
            )
            - np.min(
                command_series[ramp]
            )
        )
        for ramp in ramp_ids
    )

    checks = {
        "mainline_mass_residual": (
            compute_mainline_mass_residual(
                history
            )
        ),

        "ramp_mass_residual": (
            totals[
                "ramp_mass_residual"
            ]
        ),

        "boundary_mass_residual": (
            compute_upstream_boundary_mass_residual(
                history
            )
        ),

        "external_entry_mass_residual": (
            compute_external_entry_mass_residual(
                history
            )
        ),

        "max_sending_violation": (
            compute_max_sending_violation(
                history
            )
        ),

        "max_receiving_violation": (
            compute_max_receiving_violation(
                history
            )
        ),

        "max_avila_merge_violation": (
            compute_max_merge_violation(
                history
            )
        ),

        "objective_reconstruction_error": (
            totals[
                "objective_reconstruction_error"
            ]
        ),

        "max_command_time_variation": (
            max_command_time_variation
        ),
    }

    residual_names = [
        "mainline_mass_residual",
        "ramp_mass_residual",
        "boundary_mass_residual",
        "external_entry_mass_residual",
        "objective_reconstruction_error",
        "max_command_time_variation",
    ]

    for name in residual_names:
        if abs(
            checks[name]
        ) > 1e-8:
            raise AssertionError(
                f"{label}: {name} failed "
                f"with {checks[name]}."
            )

    violation_names = [
        "max_sending_violation",
        "max_receiving_violation",
        "max_avila_merge_violation",
    ]

    for name in violation_names:
        if checks[name] > 1e-8:
            raise AssertionError(
                f"{label}: {name} failed "
                f"with {checks[name]}."
            )

    return pd.DataFrame([
        {
            "policy": label,
            "check": name,
            "value": value,
        }
        for name, value in checks.items()
    ])


In [6]:
# 6. Benchmark reproduction check (fully-open constants).

fully_open_rate_vector = np.array(
    [
        float(
            S[
                "fully_open_command_by_ramp"
            ][ramp]
        )
        for ramp in ramp_ids
    ],
    dtype=float
)

fully_open_command_series = (
    build_constant_command_series(
        fully_open_rate_vector
    )
)

reproduction_history = (
    simulate_constant_rate_policy(
        fully_open_rate_vector
    )
)

reproduction_totals = (
    aggregate_exact_history(
        reproduction_history
    )
)

official_history_totals = (
    aggregate_exact_history(
        S[
            "official_benchmark_history"
        ]
    )
)

comparison_keys = [
    "mainline_tts",
    "ramp_queue_tts",
    "upstream_boundary_tts",
    "external_entry_tts",
    "fairness_penalty",
    "safe_exposure",
    "spillback_exposure",
    "weighted_safe_penalty",
    "weighted_spillback_penalty",
    "raw_objective",
    "final_total_inventory",
]

reproduction_rows = []

for key in comparison_keys:
    reproduction_rows.append({
        "metric": key,

        "fixed_policy_wrapper": (
            reproduction_totals[key]
        ),

        "official_benchmark": (
            official_history_totals[key]
        ),

        "difference": (
            reproduction_totals[key]
            - official_history_totals[key]
        ),
    })

reproduction_check_df = pd.DataFrame(
    reproduction_rows
)

maximum_reproduction_error = float(
    reproduction_check_df[
        "difference"
    ].abs().max()
)

display(
    reproduction_check_df.round(12)
)

print(
    "Maximum reproduction error:",
    maximum_reproduction_error
)

if maximum_reproduction_error > 1e-8:
    raise AssertionError(
        "The fixed-policy wrapper does not reproduce "
        "the corrected benchmark."
    )

reproduction_invariants_df = (
    validate_fixed_policy_history(
        history=reproduction_history,
        command_series=(
            fully_open_command_series
        ),
        label=(
            "Fully open reproduction"
        ),
    )
)

display(
    reproduction_invariants_df.round(12)
)

print(
    "PASS: the fixed-policy wrapper reproduces "
    "the corrected benchmark exactly."
)


,metric,fixed_policy_wrapper,official_benchmark,difference
0,mainline_tts,86893.445776,86893.445776,0.0
1,ramp_queue_tts,319.342487,319.342487,0.0
2,upstream_boundary_tts,0.000000,0.000000,0.0
3,external_entry_tts,0.000000,0.000000,0.0
4,fairness_penalty,57.317955,57.317955,0.0
5,safe_exposure,2137.288992,2137.288992,0.0
6,spillback_exposure,0.000000,0.000000,0.0
7,weighted_safe_penalty,1068.644496,1068.644496,0.0
8,weighted_spillback_penalty,0.000000,0.000000,0.0
9,raw_objective,88338.750713,88338.750713,0.0


Maximum reproduction error: 0.0


,policy,check,value
0,Fully open reproduction,mainline_mass_residual,4.000000e-12
1,Fully open reproduction,ramp_mass_residual,0.000000e+00
2,Fully open reproduction,boundary_mass_residual,0.000000e+00
3,Fully open reproduction,external_entry_mass_residual,0.000000e+00
4,Fully open reproduction,max_sending_violation,0.000000e+00
5,Fully open reproduction,max_receiving_violation,0.000000e+00
6,Fully open reproduction,max_avila_merge_violation,0.000000e+00
7,Fully open reproduction,objective_reconstruction_error,1.500000e-11
8,Fully open reproduction,max_command_time_variation,0.000000e+00


PASS: the fixed-policy wrapper reproduces the corrected benchmark exactly.


In [10]:
# 7. Optimize four constant rates: DE exploration + deterministic
#    coordinate-pattern refinement + canonicalization of non-binding rates.

fixed_rate_upper_bound = (
    fully_open_rate_vector.copy()
)

if np.any(
    fixed_rate_upper_bound <= 0.0
):
    raise ValueError(
        "Fixed-rate upper bounds must be positive."
    )

log_rate_upper_bound = np.log1p(
    fixed_rate_upper_bound
)


def decode_unit_rate_vector(
    unit_vector
):
    unit_vector = np.asarray(
        unit_vector,
        dtype=float
    )

    if unit_vector.shape != (
        len(ramp_ids),
    ):
        raise ValueError(
            "Invalid normalized-rate vector."
        )

    if (
        np.any(unit_vector < 0.0)
        or np.any(unit_vector > 1.0)
    ):
        raise ValueError(
            "Normalized fixed rates must be in [0, 1]."
        )

    return np.expm1(
        unit_vector
        * log_rate_upper_bound
    )


objective_evaluations = 0

raw_rate_evaluation_cache = {}

def evaluate_raw_fixed_rate_vector(
    rate_vector
):
    global objective_evaluations

    rate_vector = validate_rate_vector(
        rate_vector
    )

    if np.any(
        rate_vector
        > fixed_rate_upper_bound + 1e-10
    ):
        raise ValueError(
            "Fixed-rate vector exceeds its "
            "mathematical upper bound."
        )

    rate_vector = np.minimum(
        rate_vector,
        fixed_rate_upper_bound
    )

    cache_key = tuple(
        np.round(
            rate_vector,
            12
        )
    )

    if cache_key not in raw_rate_evaluation_cache:
        objective_evaluations += 1

        history = simulate_constant_rate_policy(
            rate_vector
        )

        totals = aggregate_exact_history(
            history
        )

        raw_rate_evaluation_cache[
            cache_key
        ] = {
            "rate_vector": (
                rate_vector.copy()
            ),
            "objective": float(
                totals["raw_objective"]
            ),
        }

    return raw_rate_evaluation_cache[
        cache_key
    ]

def fixed_policy_objective_from_unit_vector(
    unit_vector
):
    rate_vector = decode_unit_rate_vector(
        unit_vector
    )

    result = evaluate_raw_fixed_rate_vector(
        rate_vector
    )

    return result["objective"]


def coordinate_pattern_refine(
    initial_rate_vector,
    initial_step_fraction=0.10,
    minimum_step=1e-4,
    maximum_levels=30,
    maximum_sweeps_per_level=20,
    objective_tolerance=1e-8
):
    current_rate = np.asarray(
        initial_rate_vector,
        dtype=float
    ).copy()

    current_rate = np.minimum(
        np.maximum(
            current_rate,
            0.0
        ),
        fixed_rate_upper_bound
    )

    current_result = (
        evaluate_raw_fixed_rate_vector(
            current_rate
        )
    )

    current_objective = float(
        current_result["objective"]
    )

    # Start with a meaningful rate-space step.
    step_size = np.maximum(
        0.05,
        initial_step_fraction
        * np.maximum(
            current_rate,
            1.0
        )
    )

    refinement_history = []

    for level in range(
        maximum_levels
    ):
        level_improved = False

        for sweep in range(
            maximum_sweeps_per_level
        ):
            sweep_improved = False

            for ramp_index in range(
                len(ramp_ids)
            ):
                best_rate = (
                    current_rate.copy()
                )

                best_result = (
                    current_result
                )

                for multiplier in (
                    -2.0,
                    -1.0,
                    1.0,
                    2.0,
                ):
                    candidate_rate = (
                        current_rate.copy()
                    )

                    candidate_rate[
                        ramp_index
                    ] = np.clip(
                        current_rate[
                            ramp_index
                        ]
                        + multiplier
                        * step_size[
                            ramp_index
                        ],
                        0.0,
                        fixed_rate_upper_bound[
                            ramp_index
                        ]
                    )

                    if np.allclose(
                        candidate_rate,
                        current_rate,
                        atol=1e-14,
                        rtol=0.0
                    ):
                        continue

                    candidate_result = (
                        evaluate_raw_fixed_rate_vector(
                            candidate_rate
                        )
                    )

                    if (
                        candidate_result[
                            "objective"
                        ]
                        <
                        best_result[
                            "objective"
                        ]
                        - objective_tolerance
                    ):
                        best_rate = (
                            candidate_rate
                        )

                        best_result = (
                            candidate_result
                        )

                if (
                    best_result["objective"]
                    <
                    current_objective
                    - objective_tolerance
                ):
                    current_rate = (
                        best_rate
                    )

                    current_result = (
                        best_result
                    )

                    current_objective = float(
                        current_result[
                            "objective"
                        ]
                    )

                    sweep_improved = True
                    level_improved = True

            if not sweep_improved:
                break

        refinement_history.append({
            "level": level,
            "objective": current_objective,
            "maximum_step": float(
                np.max(step_size)
            ),
            "level_improved": (
                level_improved
            ),
        })

        if not level_improved:
            step_size *= 0.5

        if float(
            np.max(step_size)
        ) <= minimum_step:
            break

    return {
    "rate_vector": (
        current_rate.copy()
    ),
    "objective": (
        current_objective
    ),
    "final_step_size": (
        step_size.copy()
    ),
    "refinement_history": (
        refinement_history
    ),
}


unit_bounds = [
    (0.0, 1.0)
    for _ in ramp_ids
]

search_seeds = [
    7,
    31,
]

search_results = []

search_start = time.perf_counter()

for seed in search_seeds:
    global_result = differential_evolution(
        fixed_policy_objective_from_unit_vector,
        bounds=unit_bounds,
        seed=seed,
        maxiter=30,
        popsize=8,
        tol=1e-6,
        atol=1e-6,
        polish=False,
        workers=1,
        updating="immediate",
        disp=False,
    )

    global_rate_vector = (
        decode_unit_rate_vector(
            global_result.x
        )
    )

    refinement_result = (
        coordinate_pattern_refine(
            global_rate_vector
        )
    )

    search_results.append({
        "seed": seed,

        "rate_vector": (
            refinement_result[
                "rate_vector"
            ]
        ),

        "objective": (
            refinement_result[
                "objective"
            ]
        ),



        "final_step_size": (
            refinement_result[
                "final_step_size"
            ]
        ),

        "method": (
            "Differential evolution "
            "+ coordinate pattern refinement"
        ),

        "global_success": (
            bool(global_result.success)
        ),

        "global_message": (
            str(global_result.message)
        ),
    })

fully_open_result = (
    evaluate_raw_fixed_rate_vector(
        fully_open_rate_vector
    )
)

search_results.append({
    "seed": None,

    "rate_vector": (
        fully_open_rate_vector.copy()
    ),

    "objective": (
        fully_open_result["objective"]
    ),



    "final_step_size": np.zeros(
        len(ramp_ids),
        dtype=float
    ),

    "method": (
        "Fully open feasible candidate"
    ),

    "global_success": True,

    "global_message": (
        "Official benchmark"
    ),
})

best_search_result = min(
    search_results,
    key=lambda item: item["objective"]
)


# Canonicalize non-binding rates: report the smallest constant command
# that reproduces the same requests, then refine once more.

def canonicalize_constant_rate_vector(
    rate_vector,
    history
):
    rate_vector = validate_rate_vector(
        rate_vector
    )

    # For a constant command c:
    #
    # requested_release = min(c, available_demand).
    #
    # max(requested_release) is therefore the smallest
    # constant command that reproduces the same requests.
    canonical_rate_vector = np.array(
        [
            max(
                float(
                    history[
                        "requested_release"
                    ][step][ramp]
                )
                for step in range(
                    num_steps
                )
            )
            for ramp in ramp_ids
        ],
        dtype=float
    )

    if np.any(
        canonical_rate_vector
        - rate_vector
        > 1e-8
    ):
        raise AssertionError(
            "Canonical rate unexpectedly exceeds "
            "the original fixed rate."
        )

    return canonical_rate_vector


precanonical_rate_vector = np.asarray(
    best_search_result[
        "rate_vector"
    ],
    dtype=float
)

precanonical_history = (
    simulate_constant_rate_policy(
        precanonical_rate_vector
    )
)

canonical_rate_vector = (
    canonicalize_constant_rate_vector(
        rate_vector=(
            precanonical_rate_vector
        ),
        history=(
            precanonical_history
        )
    )
)

# Refine once more after removing flat,
# non-identifiable command excess.
final_refinement_result = (
    coordinate_pattern_refine(
        initial_rate_vector=(
            canonical_rate_vector
        ),
        initial_step_fraction=0.02,
        minimum_step=1e-4,
        maximum_levels=30
    )
)

final_precanonical_rate_vector = (
    final_refinement_result[
        "rate_vector"
    ]
)

final_precanonical_history = (
    simulate_constant_rate_policy(
        final_precanonical_rate_vector
    )
)

best_fixed_rate_vector = (
    canonicalize_constant_rate_vector(
        rate_vector=(
            final_precanonical_rate_vector
        ),
        history=(
            final_precanonical_history
        )
    )
)

best_fixed_result = (
    evaluate_raw_fixed_rate_vector(
        best_fixed_rate_vector
    )
)

best_fixed_history_precomputed = (
    simulate_constant_rate_policy(
        best_fixed_rate_vector
    )
)

best_fixed_totals_precomputed = (
    aggregate_exact_history(
        best_fixed_history_precomputed
    )
)


# Verify that canonicalization did not change the physical trajectory.

precanonical_objective = float(
    final_refinement_result[
        "objective"
    ]
)

canonical_objective = float(
    best_fixed_result[
        "objective"
    ]
)

canonical_objective_error = abs(
    canonical_objective
    - precanonical_objective
)

maximum_release_difference = max(
    abs(
        float(
            final_precanonical_history[
                "actual_release"
            ][step][ramp]
        )
        - float(
            best_fixed_history_precomputed[
                "actual_release"
            ][step][ramp]
        )
    )
    for step in range(num_steps)
    for ramp in ramp_ids
)

maximum_state_difference = max(
    abs(
        float(
            final_precanonical_history[
                "x"
            ][step][cell]
        )
        - float(
            best_fixed_history_precomputed[
                "x"
            ][step][cell]
        )
    )
    for step in range(num_steps)
    for cell in cell_ids
)

if canonical_objective_error > 1e-8:
    raise AssertionError(
        "Canonicalization changed the objective."
    )

if maximum_release_difference > 1e-8:
    raise AssertionError(
        "Canonicalization changed accepted releases."
    )

if maximum_state_difference > 1e-8:
    raise AssertionError(
        "Canonicalization changed the mainline state."
    )


best_fixed_rate_by_ramp = (
    rate_vector_to_dictionary(
        best_fixed_rate_vector
    )
)

search_seconds = (
    time.perf_counter()
    - search_start
)

fixed_rate_table = pd.DataFrame([
    {
        "ramp": ramp,

        "constant_request_cap_veh_per_15sec": (
            best_fixed_rate_vector[index]
        ),

        "equivalent_request_cap_veh_per_hour": (
            240.0
            * best_fixed_rate_vector[index]
        ),
    }
    for index, ramp in enumerate(
        ramp_ids
    )
])

print(
    "Fixed-policy search seconds:",
    search_seconds
)

print(
    "Objective evaluations (unique):",
    objective_evaluations
)

print(
    "Selected method:",
    best_search_result["method"]
)

print(
    "Best objective after refinement + canonicalization:",
    canonical_objective
)

display(
    fixed_rate_table.round(6)
)


Fixed-policy search seconds: 252.9470781
Objective evaluations (unique): 2614
Selected method: Differential evolution + coordinate pattern refinement
Best objective after refinement + canonicalization: 88319.69626859849


,ramp,constant_request_cap_veh_per_15sec,equivalent_request_cap_veh_per_hour
0,u_4th,11.253104,2700.744898
1,u_price,14.769074,3544.577812
2,u_mattie,3.060000,734.400000
3,u_avila,2.219721,532.732993


In [11]:
# 8. Exact comparison with the benchmark.

best_fixed_command_series = (
    build_constant_command_series(
        best_fixed_rate_vector
    )
)

best_fixed_history = (
    best_fixed_history_precomputed
)

best_fixed_totals = (
    best_fixed_totals_precomputed
)

best_fixed_invariants_df = (
    validate_fixed_policy_history(
        history=best_fixed_history,
        command_series=(
            best_fixed_command_series
        ),
        label=(
            "Best fixed-rate vector found"
        ),
    )
)

display(
    best_fixed_invariants_df.round(12)
)

benchmark_totals = (
    aggregate_exact_history(
        S[
            "official_benchmark_history"
        ]
    )
)

comparison_rows = [
    {
        "policy": (
            "Fully open / no control"
        ),
        **benchmark_totals,
    },

    {
        "policy": (
            "Best fixed-rate vector found"
        ),
        **best_fixed_totals,
    },
]

comparison_df = pd.DataFrame(
    comparison_rows
)

benchmark_objective = float(
    benchmark_totals[
        "raw_objective"
    ]
)

comparison_df[
    "objective_change_vs_benchmark"
] = (
    comparison_df[
        "raw_objective"
    ]
    - benchmark_objective
)

comparison_df[
    "objective_change_pct"
] = (
    100.0
    * comparison_df[
        "objective_change_vs_benchmark"
    ]
    / benchmark_objective
)

comparison_columns = [
    "policy",
    "mainline_tts",
    "ramp_queue_tts",
    "upstream_boundary_tts",
    "external_entry_tts",
    "spillback_exposure",
    "safe_exposure",
    "fairness_penalty",
    "weighted_spillback_penalty",
    "weighted_safe_penalty",
    "raw_objective",
    "objective_change_vs_benchmark",
    "objective_change_pct",
    "final_mainline_inventory",
    "final_ramp_queue",
    "final_upstream_boundary_queue",
    "final_external_entry_queue",
    "final_total_inventory",
    "total_actual_release",
    "served_fraction",
]

display(
    comparison_df[
        comparison_columns
    ].round(6)
)

print(
    "Benchmark objective:",
    benchmark_totals[
        "raw_objective"
    ]
)

print(
    "Best fixed-rate objective:",
    best_fixed_totals[
        "raw_objective"
    ]
)

print(
    "Objective difference:",
    best_fixed_totals[
        "raw_objective"
    ]
    - benchmark_totals[
        "raw_objective"
    ]
)


,policy,check,value
0,Best fixed-rate vector found,mainline_mass_residual,6.000000e-12
1,Best fixed-rate vector found,ramp_mass_residual,0.000000e+00
2,Best fixed-rate vector found,boundary_mass_residual,0.000000e+00
3,Best fixed-rate vector found,external_entry_mass_residual,0.000000e+00
4,Best fixed-rate vector found,max_sending_violation,0.000000e+00
5,Best fixed-rate vector found,max_receiving_violation,0.000000e+00
6,Best fixed-rate vector found,max_avila_merge_violation,0.000000e+00
7,Best fixed-rate vector found,objective_reconstruction_error,-1.500000e-11
8,Best fixed-rate vector found,max_command_time_variation,0.000000e+00


,policy,mainline_tts,ramp_queue_tts,upstream_boundary_tts,external_entry_tts,spillback_exposure,safe_exposure,fairness_penalty,weighted_spillback_penalty,weighted_safe_penalty,raw_objective,objective_change_vs_benchmark,objective_change_pct,final_mainline_inventory,final_ramp_queue,final_upstream_boundary_queue,final_external_entry_queue,final_total_inventory,total_actual_release,served_fraction
0,Fully open / no control,86893.445776,319.342487,0.0,0.0,0.0,2137.288992,57.317955,0.0,1068.644496,88338.750713,0.000000,0.00000,915.076524,0.0,0.0,0.0,915.076524,4387.2,1.0
1,Best fixed-rate vector found,86420.680571,755.877024,0.0,0.0,0.0,2114.032697,86.122325,0.0,1057.016349,88319.696269,-19.054445,-0.02157,915.076299,0.0,0.0,0.0,915.076299,4387.2,1.0


Benchmark objective: 88338.75071339465
Best fixed-rate objective: 88319.69626859849
Objective difference: -19.05444479615835


In [12]:
# 9. Keep results in memory.

fixed_policy_results = {
    "policy_definition": (
        "One constant meter-command rate "
        "per ramp for all 480 steps"
    ),

    "arrival_multiplier": (
        arrival_multiplier
    ),

    "best_fixed_rate_vector": (
        best_fixed_rate_vector
    ),

    "best_fixed_rate_by_ramp": (
        best_fixed_rate_by_ramp
    ),

    "best_fixed_history": (
        best_fixed_history
    ),

    "best_fixed_totals": (
        best_fixed_totals
    ),

    "benchmark_totals": (
        benchmark_totals
    ),

    "comparison_df": (
        comparison_df
    ),

    "invariant_checks": (
        best_fixed_invariants_df
    ),

    "search_results": (
        search_results
    ),
}

print(
    "Done. No fixed-policy result file "
    "was written."
)


Done. No fixed-policy result file was written.
